# Synthetic t-T-B Transition-Network Demo

This notebook demonstrates how disconnectivity trees, barrier matrices, relaxation-time matrices, and probability evolution can be paired for a micromagnetic-style grain. The data are synthetic but the object shapes match the intended Merrill.jl/NEB adapter direction.

In [7]:
using Pkg
example_dir = basename(pwd()) == "examples" ? pwd() : joinpath(pwd(), "examples")
if !isfile(joinpath(example_dir, "Project.toml"))
    example_dir = pwd()
end
Pkg.activate(example_dir)
Pkg.develop(path=abspath(joinpath(example_dir, "..")))
Pkg.instantiate()

using DisconnectivityGraphs
using LinearAlgebra
using PlotlyJS
using Statistics

struct NotebookPlot{P}
    plot::P
end

function html_escape(value::AbstractString)
    return replace(value,
        "&" => "&amp;",
        "\"" => "&quot;",
        "<" => "&lt;",
        ">" => "&gt;")
end

Base.show(io::IO, ::MIME"text/plain", ::NotebookPlot) = print(io, "Plotly interactive figure")
Base.show(io::IO, mime::MIME"application/vnd.plotly.v1+json", value::NotebookPlot) = show(io, mime, value.plot)
function Base.show(io::IO, mime::MIME"text/html", value::NotebookPlot)
    html = sprint(show, mime, value.plot)
    print(io, """
    <iframe srcdoc="$(html_escape(html))"
            style="width:100%; height:680px; border:0;"
            sandbox="allow-scripts allow-same-origin">
    </iframe>
    """)
end

show_plotly(plot) = NotebookPlot(plot)

  Activating project at `~/Github/DisconnectivityGraphs.jl/examples`
   Resolving package versions...
  No Changes to `~/Github/DisconnectivityGraphs.jl/examples/Project.toml`
  No Changes to `~/Github/DisconnectivityGraphs.jl/examples/Manifest.toml`


show_plotly (generic function with 1 method)

## A Small Synthetic Grain Network

Energies are in joules. Moments are synthetic magnetic moments in A m². The saddle energies are zero-field values; a field-adjusted view applies a simple Zeeman correction to the minima, which is enough for visualization tests but not a substitute for a full Merrill.jl energy evaluation.

In [8]:
const kB = 1.380649e-23
const τ0 = 1.0e-9

minima = [
    Minimum(:A_plus, 0.00e-19; metadata=Dict(:moment => [5.0e-17, 0.2e-17, 0.0])),
    Minimum(:B_tilt, 0.20e-19; metadata=Dict(:moment => [3.2e-17, 2.0e-17, 0.4e-17])),
    Minimum(:C_side, 0.32e-19; metadata=Dict(:moment => [0.3e-17, 4.0e-17, 0.7e-17])),
    Minimum(:D_minus, 0.05e-19; metadata=Dict(:moment => [-5.0e-17, -0.1e-17, 0.0])),
]

saddles = [
    Saddle(:A_plus, :B_tilt, 3.6e-19),
    Saddle(:B_tilt, :C_side, 4.1e-19),
    Saddle(:C_side, :D_minus, 4.0e-19),
    Saddle(:A_plus, :D_minus, 6.8e-19),
]

landscape = LandscapeGraph(minima, saddles)
tree = disconnectivity_tree(landscape)

DisconnectivityTree{Symbol, Float64}(DisconnectivityNode{Symbol, Float64}[DisconnectivityNode{Symbol, Float64}(1, 0.0, Int64[], [:A_plus]), DisconnectivityNode{Symbol, Float64}(2, 2.0e-20, Int64[], [:B_tilt]), DisconnectivityNode{Symbol, Float64}(3, 3.2e-20, Int64[], [:C_side]), DisconnectivityNode{Symbol, Float64}(4, 5.0e-21, Int64[], [:D_minus]), DisconnectivityNode{Symbol, Float64}(5, 3.6e-19, [1, 2], [:A_plus, :B_tilt]), DisconnectivityNode{Symbol, Float64}(6, 4.0e-19, [3, 4], [:C_side, :D_minus]), DisconnectivityNode{Symbol, Float64}(7, 4.1e-19, [5, 6], [:A_plus, :B_tilt, :C_side, :D_minus])], 7, Dict(:D_minus => 4, :C_side => 3, :B_tilt => 2, :A_plus => 1))

## Cooling and Reversal Histories

The stable-field history keeps the field fixed. The reversal history uses the same peak field but rotates the direction over a finite interval. This mirrors the eventual notebook use case: compare stable-field and reversing-field probability flow without changing field strength.

In [9]:
years_to_seconds(y) = y * 365.25 * 24 * 3600
smoothstep(x) = x <= 0 ? 0.0 : x >= 1 ? 1.0 : x^2 * (3 - 2x)

duration_yr = 1.0e6
reversal_duration_yr = 3.0e4
times_yr = range(0, duration_yr; length=220)
temperature_C(t) = 570.0 - 550.0 * (t / duration_yr)

function stable_field(t; B_uT=50.0)
    return 1e-6 * B_uT .* [1.0, 0.0, 0.0]
end

function reversal_field(t; B_uT=50.0)
    center = 0.48 * duration_yr
    start = center - reversal_duration_yr / 2
    ξ = smoothstep((t - start) / reversal_duration_yr)
    θ = π * ξ
    return 1e-6 * B_uT .* [cos(θ), sin(θ), 0.0]
end

function plot_histories()
    T = temperature_C.(times_yr)
    Bx_stable = [stable_field(t)[1] * 1e6 for t in times_yr]
    Bx_reversal = [reversal_field(t)[1] * 1e6 for t in times_yr]
    By_reversal = [reversal_field(t)[2] * 1e6 for t in times_yr]
    show_plotly(Plot([
        scatter(x=times_yr ./ 1e6, y=T, mode="lines", name="temperature (C)", yaxis="y1"),
        scatter(x=times_yr ./ 1e6, y=Bx_stable, mode="lines", name="stable Bx (uT)", yaxis="y2"),
        scatter(x=times_yr ./ 1e6, y=Bx_reversal, mode="lines", name="reversal Bx (uT)", yaxis="y2"),
        scatter(x=times_yr ./ 1e6, y=By_reversal, mode="lines", name="reversal By (uT)", yaxis="y2"),
    ], Layout(title="Synthetic t-T-B histories", xaxis=attr(title="time (Myr)"),
        yaxis=attr(title="temperature (C)"), yaxis2=attr(title="field (uT)", overlaying="y", side="right"),
        width=850, height=460))
    )
end

plot_histories()

Plotly interactive figure

## Barrier and Relaxation Matrices

For state `i -> j`, the barrier is `E_saddle(i,j) - E_i`. The relaxation-time matrix shows `log10(τ/year)` at a representative temperature. In real Merrill.jl integration this would come from `field_adjusted_barrier_matrix` and the NEB-derived transition network.

In [10]:
ids = minimum_ids(landscape)
minimum_by_id = Dict(m.id => m for m in landscape.minima)
index = Dict(id => i for (i, id) in pairs(ids))

function barrier_matrix(landscape)
    n = length(landscape.minima)
    B = fill(Inf, n, n)
    for s in landscape.saddles
        i, j = index[s.from], index[s.to]
        B[i, j] = s.energy - minimum_by_id[s.from].energy
        B[j, i] = s.energy - minimum_by_id[s.to].energy
    end
    B
end

function field_adjusted_minimum_energy(minimum, B_T)
    minimum.energy - dot(minimum.metadata[:moment], B_T)
end

function field_adjusted_barriers(landscape, B_T)
    n = length(landscape.minima)
    adjusted = fill(Inf, n, n)
    for s in landscape.saddles
        i, j = index[s.from], index[s.to]
        adjusted[i, j] = s.energy - field_adjusted_minimum_energy(minimum_by_id[s.from], B_T)
        adjusted[j, i] = s.energy - field_adjusted_minimum_energy(minimum_by_id[s.to], B_T)
    end
    adjusted
end

function relaxation_log10_years(barriers, T_K)
    years = 365.25 * 24 * 3600
    out = similar(barriers)
    for i in axes(barriers, 1), j in axes(barriers, 2)
        out[i, j] = isfinite(barriers[i, j]) ? log10(τ0) + barriers[i, j] / (kB * T_K) / log(10) - log10(years) : NaN
    end
    out
end

Trep_K = 570.0 + 273.15
Brep = reversal_field(0.50 * duration_yr)
barriers = field_adjusted_barriers(landscape, Brep)
relax = relaxation_log10_years(barriers, Trep_K)

function plotly_finite_matrix(mat)
    out = Matrix{Union{Nothing,Float64}}(undef, size(mat))
    for i in axes(mat, 1), j in axes(mat, 2)
        v = Float64(mat[i, j])
        out[i, j] = isfinite(v) ? v : nothing
    end
    out
end

barrier_z = plotly_finite_matrix(barriers ./ (kB * Trep_K))
relax_z = plotly_finite_matrix(relax)

show_plotly(Plot([
    heatmap(z=barrier_z, x=String.(ids), y=String.(ids), colorscale="Viridis", colorbar=attr(title="E/kBT"))
], Layout(title="Field-adjusted transition barriers at 570 C", xaxis=attr(title="to state"), yaxis=attr(title="from state"), width=620, height=520)))

Plotly interactive figure

In [11]:
show_plotly(Plot([
    heatmap(z=relax_z, x=String.(ids), y=String.(ids), colorscale="Plasma", colorbar=attr(title="log10 yr"))
], Layout(title="Synthetic transition relaxation times at 570 C", xaxis=attr(title="to state"), yaxis=attr(title="from state"), width=620, height=520)))

Plotly interactive figure

## Lightweight Probability Evolution

This is a compact demonstration of the same concept used in the reversal-paleointensity notebooks: build a Markov generator from barriers at each time step, propagate occupancy, and compare stable-field versus reversal-field final alignment.

In [12]:
function generator_from_barriers(barriers, T_K)
    n = size(barriers, 1)
    Q = zeros(n, n)
    for from in 1:n, to in 1:n
        from == to && continue
        if isfinite(barriers[from, to])
            rate = inv(τ0) * exp(-barriers[from, to] / (kB * T_K))
            Q[to, from] = rate
            Q[from, from] -= rate
        end
    end
    Q
end

function evolve(field_function)
    rho = zeros(length(ids)); rho[index[:A_plus]] = 1.0
    traces = [(t=0.0, rho=copy(rho))]
    for (t0, t1) in zip(times_yr[1:end-1], times_yr[2:end])
        tm = 0.5 * (t0 + t1)
        T_K = temperature_C(tm) + 273.15
        Q = generator_from_barriers(field_adjusted_barriers(landscape, field_function(tm)), T_K)
        rho = exp(Q * years_to_seconds(t1 - t0)) * rho
        rho = max.(rho, 0); rho ./= sum(rho)
        push!(traces, (t=t1, rho=copy(rho)))
    end
    traces
end

stable_trace = evolve(stable_field)
reversal_trace = evolve(reversal_field)

function plot_occupancy(stable_trace, reversal_trace)
    traces = PlotlyJS.AbstractTrace[]
    for (case, tr) in [("stable", stable_trace), ("reversal", reversal_trace)]
        for (i, id) in pairs(ids)
            push!(traces, scatter(x=[r.t / 1e6 for r in tr], y=[r.rho[i] for r in tr], mode="lines", name="$case $id"))
        end
    end
    show_plotly(Plot(traces, Layout(title="Synthetic probability evolution", xaxis=attr(title="time (Myr)"), yaxis=attr(title="state probability", range=[0, 1]), width=900, height=520)))
end

plot_occupancy(stable_trace, reversal_trace)

Plotly interactive figure